# CNN inverse training on Colab T4

Workflow: clone repo from GitHub → install deps → generate dataset → train CNN → evaluate.

**Auth note:** the repo is private. Either make it public (simplest), or add a `GH_TOKEN` Colab secret (🔑 icon in sidebar) with a GitHub PAT that has `repo` scope.

To iterate on hyperparameters: edit `notebooks/train_cnn_inverse.py` locally → `git push` → in this notebook run a `!cd /content/water_v2 && git pull` cell → re-run the training cell.

In [ ]:
# 1) Clone repo, install deps, verify GPU
import os, sys, subprocess

REPO_DIR = "/content/water_v2"
REPO_URL_PUBLIC = "https://github.com/alexhrubin/water_v2.git"

if not os.path.isdir(REPO_DIR):
    # Try public clone first; fall back to token-authenticated clone if private
    r = subprocess.run(["git", "clone", "-b", "python-rewrite", REPO_URL_PUBLIC, REPO_DIR],
                       capture_output=True, text=True)
    if r.returncode != 0:
        try:
            from google.colab import userdata
            token = userdata.get("GH_TOKEN")
        except Exception:
            token = None
        if not token:
            raise RuntimeError(
                "Public clone failed and no GH_TOKEN secret found.\n"
                "Either make the repo public, or add a GH_TOKEN Colab secret "
                "(🔑 icon in sidebar, value = a GitHub PAT with `repo` scope).\n\n"
                f"Git error:\n{r.stderr}"
            )
        url_auth = f"https://{token}@github.com/alexhrubin/water_v2.git"
        subprocess.run(["git", "clone", "-b", "python-rewrite", url_auth, REPO_DIR], check=True)
    print(f"Cloned to {REPO_DIR}")
else:
    print(f"{REPO_DIR} already exists; skipping clone. Run `!cd {REPO_DIR} && git pull` to update.")

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.run(["pip", "install", "-q", "equinox", "optax"], check=True)

import jax
print(f"JAX backend: {jax.default_backend()}  devices: {jax.devices()}")
from wavetank import Tank, build_propagator   # noqa: F401
print("wavetank imports OK")

In [ ]:
# Optional: pull latest changes if you've pushed updates since cloning
!cd /content/water_v2 && git pull

In [ ]:
# 2) Generate 1M-sample dataset on GPU (~1 min on T4)
# Writes to data/naive_inverse/dataset.npz (~17 GB) inside Colab session storage.
!python -u notebooks/gen_naive_inverse_data.py

In [ ]:
# 3) Train CNN (~5-10 min on T4)
!python -u notebooks/train_cnn_inverse.py

In [ ]:
# 4) OOD eval
!python -u notebooks/eval_cnn_inverse.py

In [ ]:
from IPython.display import Image, display
display(Image("data/naive_inverse/eval_cnn_ood.png"))